# K-Means Clustering from Scratch

## 1. Load and Visualize the Dataset

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.io

# Download the dataset
import urllib.request
url = "https://github.com/krasserm/machine-learning-notebooks/raw/master/data/ml-ex7/ex7data2.mat"
urllib.request.urlretrieve(url, "ex7data2.mat")

mat = scipy.io.loadmat("ex7data2.mat")
X = mat["X"]

print(f"Dataset shape: {X.shape}")

plt.figure(figsize=(7, 5))
plt.scatter(X[:, 0], X[:, 1], s=15, color="steelblue", alpha=0.7)
plt.title("ex7data2 — Raw Data")
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.tight_layout()
plt.show()

## 2. Find Closest Centroids

In [ ]:
def find_closest_centroids(X, centroids):
    """
    Assigns each data point in X to the index of the closest centroid.
    Uses Euclidean distance.

    Parameters:
        X         : (m, n) array of data points
        centroids : (K, n) array of centroid positions

    Returns:
        idx : (m,) array of centroid indices (0-indexed)
    """
    m = X.shape[0]
    idx = np.zeros(m, dtype=int)

    for i in range(m):
        distances = np.linalg.norm(X[i] - centroids, axis=1)
        idx[i] = np.argmin(distances)

    return idx


initial_centroids = np.array([[3, 3], [6, 2], [8, 5]])
idx = find_closest_centroids(X, initial_centroids)

print("Closest centroid index for the first 3 data points:")
for i in range(3):
    print(f"  Point {i}: {X[i]} -> Centroid {idx[i]}")

## 3. Compute New Centroids

In [ ]:
def compute_centroids(X, idx, K):
    """
    Computes new centroid positions as the mean of all data points
    assigned to each cluster.

    Parameters:
        X   : (m, n) array of data points
        idx : (m,) array of centroid assignments
        K   : number of clusters

    Returns:
        centroids : (K, n) array of updated centroid positions
    """
    n = X.shape[1]
    centroids = np.zeros((K, n))

    for k in range(K):
        points_in_cluster = X[idx == k]
        if len(points_in_cluster) > 0:
            centroids[k] = points_in_cluster.mean(axis=0)

    return centroids


K = 3
new_centroids = compute_centroids(X, idx, K)

print("Updated centroid positions:")
for k, c in enumerate(new_centroids):
    print(f"  Centroid {k}: {c}")

## 4. Run K-Means

In [ ]:
def run_k_means(X, initial_centroids, max_iters=10):
    """
    Runs the K-means algorithm for max_iters iterations.

    Parameters:
        X                 : (m, n) array of data points
        initial_centroids : (K, n) array of starting centroid positions
        max_iters         : number of iterations to run

    Returns:
        idx               : (m,) final cluster assignments
        centroids         : (K, n) final centroid positions
        centroid_history  : list of centroid arrays at each iteration
    """
    K = initial_centroids.shape[0]
    centroids = initial_centroids.copy()
    centroid_history = [centroids.copy()]

    for _ in range(max_iters):
        idx = find_closest_centroids(X, centroids)
        centroids = compute_centroids(X, idx, K)
        centroid_history.append(centroids.copy())

    return idx, centroids, centroid_history


idx, final_centroids, centroid_history = run_k_means(X, initial_centroids, max_iters=10)

colors = ["steelblue", "tomato", "mediumseagreen"]
markers = ["o", "s", "^"]

plt.figure(figsize=(7, 5))
for k in range(K):
    cluster_points = X[idx == k]
    plt.scatter(cluster_points[:, 0], cluster_points[:, 1],
                s=15, color=colors[k], alpha=0.6, label=f"Cluster {k}")

plt.scatter(final_centroids[:, 0], final_centroids[:, 1],
            s=180, color="black", marker="X", zorder=5, label="Final Centroids")

plt.title("K-Means Clustering Result (10 iterations)")
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.legend()
plt.tight_layout()
plt.show()

## 5. Centroid Convergence Visualization

In [ ]:
plt.figure(figsize=(7, 5))

for k in range(K):
    cluster_points = X[idx == k]
    plt.scatter(cluster_points[:, 0], cluster_points[:, 1],
                s=15, color=colors[k], alpha=0.4)

for k in range(K):
    path = np.array([c[k] for c in centroid_history])
    plt.plot(path[:, 0], path[:, 1], color=colors[k],
             linewidth=1.5, marker="o", markersize=5, label=f"Centroid {k} path")
    plt.scatter(path[0, 0], path[0, 1], color=colors[k],
                s=100, edgecolors="black", zorder=5)

plt.title("Centroid Convergence Over Iterations")
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.legend()
plt.tight_layout()
plt.show()

## 6. Initialize Centroids Randomly

In [ ]:
def init_centroids(X, K):
    """
    Randomly selects K data points from X as initial centroids.

    Parameters:
        X : (m, n) array of data points
        K : number of clusters

    Returns:
        centroids : (K, n) array of randomly selected initial centroids
    """
    indices = np.random.choice(X.shape[0], size=K, replace=False)
    return X[indices].copy()


np.random.seed(0)
random_centroids = init_centroids(X, K=3)

print("Randomly initialized centroids:")
for k, c in enumerate(random_centroids):
    print(f"  Centroid {k}: {c}")

## 7. Run K-Means with Random Initialization

In [ ]:
idx_rand, final_centroids_rand, _ = run_k_means(X, random_centroids, max_iters=10)

plt.figure(figsize=(7, 5))
for k in range(K):
    cluster_points = X[idx_rand == k]
    plt.scatter(cluster_points[:, 0], cluster_points[:, 1],
                s=15, color=colors[k], alpha=0.6, label=f"Cluster {k}")

plt.scatter(final_centroids_rand[:, 0], final_centroids_rand[:, 1],
            s=180, color="black", marker="X", zorder=5, label="Final Centroids")

plt.title("K-Means with Random Initialization")
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.legend()
plt.tight_layout()
plt.show()

## 8. Cluster Assignment Report

In [ ]:
import pandas as pd

report = pd.DataFrame({
    "Feature 1": X[:, 0],
    "Feature 2": X[:, 1],
    "Cluster": idx_rand
})

print("Cluster Assignment Report (first 10 rows):")
print(report.head(10).to_string(index=True))

print("\nCluster sizes:")
print(report["Cluster"].value_counts().sort_index().to_string())